In [1]:
import pandas as pd
import jieba
from torch import nn
import smart_open
from gensim.models import Word2Vec
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer
from transformers import BertForSequenceClassification

### 检查GPU

In [2]:
    # setting device on GPU if available, else CPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)
    print()
    
    #Additional Info when using cuda
    if device.type == 'cuda':
        print(torch.cuda.get_device_name(0))
        print('Memory Usage:')
        print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
        print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')


Using device: cuda

NVIDIA A100-SXM4-40GB
Memory Usage:
Allocated: 0.0 GB
Cached:    0.0 GB


### 数据读取和删除无用词

In [3]:
# 数据读取
def load_tsv(file_path):
    data = pd.read_csv(file_path, sep='\t')
    data_x = data.iloc[:, -1]
    data_y = data.iloc[:, 1]
    return data_x, data_y
 
train_x, train_y = load_tsv("train.tsv")
test_x, test_y = load_tsv("test.tsv")
train_x=[list(jieba.cut(x)) for x in train_x]
test_x=[list(jieba.cut(x)) for x in test_x]

# 中文停用词
with open('hit_stopwords.txt','r',encoding='UTF8') as f: # with可以在代码执行完后自动关闭文件
    stop_words=[word.strip() for word in f.readlines()]
    print('Successfully')

def drop_stopword(datas):
    for data in datas:
        for word in data:
            if word in stop_words:
                data.remove(word)
    return datas
 
def save_data(datax,path):
    with open(path, 'w', encoding="UTF8") as f:
        for lines in datax:
            for i, line in enumerate(lines):
                f.write(str(line))
                # 如果不是最后一行，就添加一个逗号
                if i != len(lines) - 1:
                    f.write(',')
            f.write('\n')

train_x=drop_stopword(train_x)
test_x=drop_stopword(test_x)
save_data(train_x,'clean_train.txt')
save_data(test_x,'clean_test.txt')
def load_txt(path):
    with open(path,'r',encoding='utf-8') as f:
        data=[[line.strip()] for line in f.readlines()]
        return data
train_x=load_txt('clean_train.txt')
test_x=load_txt('clean_test.txt')
train=train_x+test_x

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Dumping model to file cache /tmp/jieba.cache
Dump cache file failed.
Traceback (most recent call last):
  File "/nfs/home/4001_tengyishu/.local/lib/python3.6/site-packages/jieba/__init__.py", line 154, in initialize
    _replace_file(fpath, cache_file)
PermissionError: [Errno 1] Operation not permitted: '/tmp/tmpwrapy57y' -> '/tmp/jieba.cache'
Loading model cost 0.778 seconds.
Prefix dict has been built successfully.


Successfully


### 转成词向量

In [4]:
X_all=[i for x in train for i in x]
word2vec_model = Word2Vec(sentences=X_all, size=100, window=5, min_count=1, workers=4)
# 将文本转换为Word2Vec向量表示
def text_to_vector(text):
    vector = [word2vec_model.wv[word] for word in text if word in word2vec_model.wv]
    return np.mean(vector, axis=0) if vector else [0] * word2vec_model.vector_size
 
X_train_w2v = [[text_to_vector(text)] for line in train_x for text in line]
X_test_w2v = [[text_to_vector(text)] for line in test_x for text in line]

### 词向量转换为loader

In [5]:
# 将词向量转换为PyTorch张量
X_train_array = np.array(X_train_w2v, dtype=np.float32)
X_train_tensor = torch.Tensor(X_train_array)
X_test_array = np.array(X_test_w2v, dtype=np.float32)
X_test_tensor = torch.Tensor(X_test_array)
#使用DataLoader打包文件
#设置LongTensor是因为分类任务很多损失函数需要
train_dataset = TensorDataset(X_train_tensor, torch.LongTensor(train_y))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataset = TensorDataset(X_test_tensor,torch.LongTensor(test_y))
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=True)

In [6]:
print('训练集data维数：',X_train_tensor.shape)
print('训练集tag维数：',train_y.shape)
print('测试集data维数：',X_test_tensor.shape)
print('测试集tag维数：',test_y.shape)

训练集data维数： torch.Size([56700, 1, 100])
训练集tag维数： (56700,)
测试集data维数： torch.Size([6300, 1, 100])
测试集tag维数： (6300,)


## LSTM

### 模型定义

In [10]:
# 定义LSTM模型
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
 
    def forward(self, x):# x为输入向量
        lstm_out, _ = self.lstm(x)# lstm_out为lstm的输出，_为隐状态
        output = self.fc(lstm_out[:, -1, :])  # 取序列的最后一个输出（本来也就一个）
        return output
    
input_size = word2vec_model.vector_size
hidden_size = 50  # 你可以根据需要调整隐藏层大小
output_size = 2  # 输出的维度，lstm中y都是二维的
 
model = LSTMModel(input_size, hidden_size, output_size)
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 交叉熵损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

### 模型训练

In [11]:
num_epochs = 10
log_interval = 100  # 每隔100个批次输出一次日志
loss_min=100

# 定义变量来跟踪验证集上的最小损失
best_loss = np.inf
patience = 3  # 定义耐心值，即连续几个epoch验证集损失不减少时停止训练
count = 0
losses=[]
for epoch in range(num_epochs):
    model.train()  # 设置模型为训练模式
    for batch_idx, (data, target) in enumerate(train_loader):
        outputs = model(data)
        loss = criterion(outputs, target)
 
        optimizer.zero_grad()# 清除梯度值，以便于下一次计算梯度
        loss.backward()# 反向传播算法计算损失函数关于模型参数的梯度
        optimizer.step()# 根据梯度和学习率来更新模型参数
 
        if batch_idx % log_interval == 0:
            print('Epoch [{}/{}], Batch [{}/{}], Loss: {:.4f}'.format(
                epoch + 1, num_epochs, batch_idx, len(train_loader), loss.item()))
        losses.append(loss.item())
    
#     # 在每个epoch结束后，评估模型在验证集上的性能
#     model.eval()
#     valid_loss = 0.0
#     with torch.no_grad():
#         for data, target in test_loader:
#             outputs = model(data)
#             valid_loss += criterion(outputs, target).item()

#     valid_loss /= len(test_loader)
#     print('Validation Loss: {:.4f}'.format(valid_loss))
    
#     # 如果验证集损失下降，保存模型并更新最佳损失
#     if valid_loss < best_loss:
#         best_loss = valid_loss
#         count = 0  # 重置计数器
#         torch.save(model.state_dict(), 'best_model.pth')
#     else:
#         # 如果验证集损失没有下降，增加计数器
#         count += 1
#         # 如果计数器达到耐心值，停止训练
#         if count >= patience:
#             print('Early stopping: no improvement for {} consecutive epochs.'.format(patience))
#             break


Epoch [1/10], Batch [0/443], Loss: 0.6887
Epoch [1/10], Batch [100/443], Loss: 0.3831
Epoch [1/10], Batch [200/443], Loss: 0.3259
Epoch [1/10], Batch [300/443], Loss: 0.2704
Epoch [1/10], Batch [400/443], Loss: 0.2720
Epoch [2/10], Batch [0/443], Loss: 0.3065
Epoch [2/10], Batch [100/443], Loss: 0.2611
Epoch [2/10], Batch [200/443], Loss: 0.1804
Epoch [2/10], Batch [300/443], Loss: 0.2883
Epoch [2/10], Batch [400/443], Loss: 0.2969
Epoch [3/10], Batch [0/443], Loss: 0.2399
Epoch [3/10], Batch [100/443], Loss: 0.2550
Epoch [3/10], Batch [200/443], Loss: 0.2905
Epoch [3/10], Batch [300/443], Loss: 0.2070
Epoch [3/10], Batch [400/443], Loss: 0.2182
Epoch [4/10], Batch [0/443], Loss: 0.2378
Epoch [4/10], Batch [100/443], Loss: 0.3324
Epoch [4/10], Batch [200/443], Loss: 0.2666
Epoch [4/10], Batch [300/443], Loss: 0.1761
Epoch [4/10], Batch [400/443], Loss: 0.1429
Epoch [5/10], Batch [0/443], Loss: 0.2121
Epoch [5/10], Batch [100/443], Loss: 0.2618
Epoch [5/10], Batch [200/443], Loss: 0.222

### 模型评估

In [12]:
with torch.no_grad():# 关掉自动求导功能
    model.eval()#设置模型为评估模式
    correct = 0
    total = 0
    for data, target in test_loader:# 这个循环适应一个batch中多列同时测试准确度
        outputs = model(data)# 得到预测结果
        _, predicted = torch.max(outputs.data, 1)# 输出张量中每行最大值的索引
        total += target.size(0)
        correct += (predicted == target).sum().item()
 
    accuracy = correct / total
    print('Test Accuracy: {:.2%}'.format(accuracy))
    

Test Accuracy: 90.86%


## Bi-LSTM

In [10]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(BiLSTMModel, self).__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)  # Multiply hidden_size by 2 because it's bidirectional
        
    def forward(self, x): # x is the input vector
        lstm_out, _ = self.bilstm(x) # lstm_out is the output of Bi-LSTM, _ is the hidden state
        # Concatenate the last forward and backward outputs
        combined_out = torch.cat((lstm_out[:, -1, :hidden_size], lstm_out[:, 0, hidden_size:]), dim=1)
        output = self.fc(combined_out) 
        return output

input_size = word2vec_model.vector_size
hidden_size = 50 # You can adjust the hidden layer size as needed
output_size = 2 # Output dimension, y in LSTM is two-dimensional

model = BiLSTMModel(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()  # 交叉熵损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

### 模型训练

In [11]:
num_epochs = 10
log_interval = 100  # 每隔100个批次输出一次日志
loss_min = 100

# 定义变量来跟踪验证集上的最小损失
best_loss = np.inf
patience = 3  # 定义耐心值，即连续几个epoch验证集损失不减少时停止训练
count = 0
losses = []

for epoch in range(num_epochs):
    model.train()  # 设置模型为训练模式
    for batch_idx, (data, target) in enumerate(train_loader):
        outputs = model(data)
        loss = criterion(outputs, target)  # 使用 nn.CrossEntropyLoss
        
        optimizer.zero_grad()  # 清除梯度值，以便于下一次计算梯度
        loss.backward()  # 反向传播算法计算损失函数关于模型参数的梯度
        optimizer.step()  # 根据梯度和学习率来更新模型参数
        
        if batch_idx % log_interval == 0:
            print('Epoch [{}/{}], Batch [{}/{}], Loss: {:.4f}'.format(
                epoch + 1, num_epochs, batch_idx, len(train_loader), loss.item()))
        losses.append(loss.item())
    
    # 在每个epoch结束后，评估模型在验证集上的性能
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            outputs = model(data)
            valid_loss += criterion(outputs, target).item()

    valid_loss /= len(test_loader)
    print('Validation Loss: {:.4f}'.format(valid_loss))
    
    # 如果验证集损失下降，保存模型并更新最佳损失
    if valid_loss < best_loss:
        best_loss = valid_loss
        count = 0  # 重置计数器
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        # 如果验证集损失没有下降，增加计数器
        count += 1
        # 如果计数器达到耐心值，停止训练
        if count >= patience:
            print('Early stopping: no improvement for {} consecutive epochs.'.format(patience))
            break

Epoch [1/10], Batch [0/443], Loss: 0.6970
Epoch [1/10], Batch [100/443], Loss: 0.3620
Epoch [1/10], Batch [200/443], Loss: 0.3020
Epoch [1/10], Batch [300/443], Loss: 0.3006
Epoch [1/10], Batch [400/443], Loss: 0.2696
Validation Loss: 0.2640
Epoch [2/10], Batch [0/443], Loss: 0.1932
Epoch [2/10], Batch [100/443], Loss: 0.3329
Epoch [2/10], Batch [200/443], Loss: 0.2458
Epoch [2/10], Batch [300/443], Loss: 0.2451
Epoch [2/10], Batch [400/443], Loss: 0.1740
Validation Loss: 0.2592
Epoch [3/10], Batch [0/443], Loss: 0.1975
Epoch [3/10], Batch [100/443], Loss: 0.3442
Epoch [3/10], Batch [200/443], Loss: 0.1789
Epoch [3/10], Batch [300/443], Loss: 0.2858
Epoch [3/10], Batch [400/443], Loss: 0.3556
Validation Loss: 0.2493
Epoch [4/10], Batch [0/443], Loss: 0.1496
Epoch [4/10], Batch [100/443], Loss: 0.2761
Epoch [4/10], Batch [200/443], Loss: 0.2395
Epoch [4/10], Batch [300/443], Loss: 0.2565
Epoch [4/10], Batch [400/443], Loss: 0.1966
Validation Loss: 0.2434
Epoch [5/10], Batch [0/443], Los

### 模型评估

In [12]:
with torch.no_grad():# 关掉自动求导功能
    model.eval()#设置模型为评估模式
    correct = 0
    total = 0
    for data, target in test_loader:# 这个循环适应一个batch中多列同时测试准确度
        outputs = model(data)# 得到预测结果
        _, predicted = torch.max(outputs.data, 1)# 输出张量中每行最大值的索引
        total += target.size(0)
        correct += (predicted == target).sum().item()
 
    accuracy = correct / total
    print('Test Accuracy: {:.2%}'.format(accuracy))

Test Accuracy: 91.02%


## Transformer

### 转换训练格式

In [3]:
def load_tsv(file_path):
    data = pd.read_csv(file_path, sep='\t')
    data_x = data.iloc[:, -1]
    data_y = data.iloc[:, 1]
    return data_x, data_y
 
train_x, train_y = load_tsv("train.tsv")
test_x, test_y = load_tsv("test.tsv")
train_x=[list(jieba.cut(x)) for x in train_x]
test_x=[list(jieba.cut(x)) for x in test_x]

# 中文停用词
with open('hit_stopwords.txt','r',encoding='UTF8') as f: # with可以在代码执行完后自动关闭文件
    stop_words=[word.strip() for word in f.readlines()]
    print('Successfully')

def drop_stopword(datas):
    for data in datas:
        for word in data:
            if word in stop_words:
                data.remove(word)
    return datas
 
def save_data(datax,path):
    with open(path, 'w', encoding="UTF8") as f:
        for lines in datax:
            for i, line in enumerate(lines):
                f.write(str(line))
                # 如果不是最后一行，就添加一个逗号
                if i != len(lines) - 1:
                    f.write(',')
            f.write('\n')

train_x = drop_stopword(train_x)
test_x = drop_stopword(test_x)
train_x = [" ".join(sentence) for sentence in train_x]
test_x = [" ".join(sentence) for sentence in test_x]

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Dumping model to file cache /tmp/jieba.cache
Dump cache file failed.
Traceback (most recent call last):
  File "/nfs/home/4001_tengyishu/.local/lib/python3.6/site-packages/jieba/__init__.py", line 154, in initialize
    _replace_file(fpath, cache_file)
PermissionError: [Errno 1] Operation not permitted: '/tmp/tmpvlygj97s' -> '/tmp/jieba.cache'
Loading model cost 0.804 seconds.
Prefix dict has been built successfully.


Successfully


### 加载loader

In [4]:

# 加载预训练的中文BERT分词器
tokenizer = BertTokenizer.from_pretrained('/nfs/home/4001_tengyishu/Desktop/微博气候/文本情感分析/')

# 对文本进行分词和编码
inputs = tokenizer(train_x, return_tensors="pt", padding=True, truncation=True, max_length=100)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

# 转换标签为张量
labels_tensor = torch.tensor(train_y)

# 创建数据加载器
dataset = TensorDataset(input_ids, attention_mask, labels_tensor)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True)

# 加载测试数据
inputs_test = tokenizer(test_x, return_tensors="pt", padding=True, truncation=True, max_length=100)
input_ids_test = inputs_test['input_ids']
attention_mask_test = inputs_test['attention_mask']
labels_test = torch.tensor(test_y)

# 创建测试数据加载器
dataset_test = TensorDataset(input_ids_test, attention_mask_test, labels_test)
test_loader = DataLoader(dataset_test, batch_size=128, shuffle=False)

In [5]:
# 加载预训练的中文BERT模型并添加分类头
model = BertForSequenceClassification.from_pretrained('/nfs/home/4001_tengyishu/Desktop/微博气候/文本情感分析/', num_labels=2)

# 定义优化器和损失函数
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

# best_loss = np.inf
# patience = 2  # 定义耐心值，即连续几个epoch验证集损失不减少时停止训练
# count = 0
losses=[]

# 训练模型
for epoch in range(1):  # 训练1个epoch
    model.train()
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        print(f'Epoch {epoch+1}, Loss: {loss.item()}')
        losses.append(loss.item())
        
#     model.eval()  # 将模型设置为评估模式
#     total_loss = 0
#     with torch.no_grad():  # 关闭梯度计算
#         for batch_idx, batch in enumerate(val_loader):  # 使用验证集数据
#             input_ids, attention_mask, labels = batch
#             outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
#             total_loss += outputs.loss.item()
#     avg_loss = total_loss / len(val_loader)  # 计算平均损失
#     print(f'Validation Loss after epoch {epoch + 1}: {avg_loss}')
    
#     # 如果验证集损失下降，保存模型并更新最佳损失
#     if valid_loss < best_loss:
#         best_loss = valid_loss
#         count = 0  # 重置计数器
#         torch.save(model.state_dict(), 'best_model.pth')
#     else:
#         # 如果验证集损失没有下降，增加计数器
#         count += 1
#         # 如果计数器达到耐心值，停止训练
#         if count >= patience:
#             print('Early stopping: no improvement for {} consecutive epochs.'.format(patience))
#             break

Some weights of the model checkpoint at /nfs/home/4001_tengyishu/Desktop/微博气候/文本情感分析/ were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized 

Epoch 1, Loss: 0.6565685868263245
Epoch 1, Loss: 0.6077581644058228
Epoch 1, Loss: 0.5405043959617615
Epoch 1, Loss: 0.4099717438220978
Epoch 1, Loss: 0.29761841893196106
Epoch 1, Loss: 0.2748761773109436
Epoch 1, Loss: 0.19674400985240936
Epoch 1, Loss: 0.17653147876262665
Epoch 1, Loss: 0.2569282352924347
Epoch 1, Loss: 0.20946559309959412
Epoch 1, Loss: 0.22026632726192474
Epoch 1, Loss: 0.28290116786956787
Epoch 1, Loss: 0.20261849462985992
Epoch 1, Loss: 0.1495022475719452
Epoch 1, Loss: 0.2746548056602478
Epoch 1, Loss: 0.24988336861133575
Epoch 1, Loss: 0.12151165306568146
Epoch 1, Loss: 0.1131778284907341
Epoch 1, Loss: 0.180583193898201
Epoch 1, Loss: 0.15331681072711945
Epoch 1, Loss: 0.14438985288143158
Epoch 1, Loss: 0.152863547205925
Epoch 1, Loss: 0.1931437849998474
Epoch 1, Loss: 0.22191555798053741
Epoch 1, Loss: 0.21034130454063416
Epoch 1, Loss: 0.1652139276266098
Epoch 1, Loss: 0.1194293200969696
Epoch 1, Loss: 0.15822355449199677
Epoch 1, Loss: 0.17267927527427673
E

Epoch 1, Loss: 0.1540093868970871
Epoch 1, Loss: 0.11473064124584198
Epoch 1, Loss: 0.03932074457406998
Epoch 1, Loss: 0.07281973212957382
Epoch 1, Loss: 0.07718972116708755
Epoch 1, Loss: 0.07311175018548965
Epoch 1, Loss: 0.028685659170150757
Epoch 1, Loss: 0.10560295730829239
Epoch 1, Loss: 0.061594631522893906
Epoch 1, Loss: 0.11573661118745804
Epoch 1, Loss: 0.18067556619644165
Epoch 1, Loss: 0.05656985566020012
Epoch 1, Loss: 0.10713440179824829
Epoch 1, Loss: 0.07434079796075821
Epoch 1, Loss: 0.07841639965772629
Epoch 1, Loss: 0.11022762954235077
Epoch 1, Loss: 0.09293793141841888
Epoch 1, Loss: 0.06868516653776169
Epoch 1, Loss: 0.16171550750732422
Epoch 1, Loss: 0.06316133588552475
Epoch 1, Loss: 0.11662370711565018
Epoch 1, Loss: 0.18101666867733002
Epoch 1, Loss: 0.11044879257678986
Epoch 1, Loss: 0.05305980145931244
Epoch 1, Loss: 0.13831840455532074
Epoch 1, Loss: 0.09952855110168457
Epoch 1, Loss: 0.13992327451705933
Epoch 1, Loss: 0.04002714902162552
Epoch 1, Loss: 0.09

In [6]:

# 测试模型
model.eval()  # 设置模型为评估模式
total_correct = 0
total_samples = 0
with torch.no_grad():  # 关闭梯度计算
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        outputs = model(input_ids, attention_mask=attention_mask)
        _, predicted = torch.max(outputs.logits, dim=1)  # 获取预测结果
        total_correct += (predicted == labels).sum().item()  # 统计预测正确的样本数
        total_samples += labels.size(0)  # 统计总样本数

# 计算准确率
accuracy = total_correct / total_samples
print(f'Test Accuracy: {accuracy}')

Test Accuracy: 0.9709523809523809
